# Phase 2: Data Cleaning


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset
We focus on the primary daily city-level dataset (`city_day.csv`) for our modeling baseline.

In [ ]:
RAW_DATA_PATH = '../data/raw/city_day.csv'
df = pd.read_csv(RAW_DATA_PATH)
print(f"Initial shape: {df.shape}")

## 2. Standardize Column Names
Convert column names to lowercase and replace spaces/dots with underscores for consistency.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('.', '_', regex=False)
print("Standardized columns:", df.columns.tolist())

## 3. Date Conversion & Sorting
Convert the `date` column to proper datetime objects and sort the data chronologically per city.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['city', 'date']).reset_index(drop=True)
print("Date format converted and data sorted.")

## 4. Remove Duplicates & Fix Inconsistent/Impossible Values
- Remove duplicate entries based on `city` and `date`.
- Air quality metrics and AQI cannot be negative. We will replace any negative values with `NaN`.

In [ ]:
# Remove duplicates
initial_len = len(df)
df = df.drop_duplicates(subset=['city', 'date'])
print(f"Removed {initial_len - len(df)} duplicate rows.")

# Handle impossible AQI and pollutant values (negatives)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df.loc[df[col] < 0, col] = np.nan

print("Inconsistent/impossible values replaced with NaN.")

## 5. Handle Missing Values


In [ ]:
def impute_missing(group):
    # Linear interpolation for gaps up to 3 days
    group = group.interpolate(method='linear', limit=3)
    # Ffill and Bfill for the rest
    group = group.ffill().bfill()
    return group

df[numeric_cols] = df.groupby('city')[numeric_cols].transform(impute_missing)

print("Missing values handled. Remaining nulls:\n", df.isnull().sum())

## 6. Detect Outliers


In [ ]:
outlier_counts = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_counts[col] = outliers

print("Detected Outliers (Retained for Modeling):")
for col, count in outlier_counts.items():
    print(f"{col}: {count}")

## 7. Save Cleaned Dataset
Save the finalized, cleaned dataset for modeling.

In [ ]:
PROCESSED_DATA_PATH = '../data/processed/clean_air_quality.csv'
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Cleaned dataset saved successfully to {PROCESSED_DATA_PATH}")